# Train/Test Predictive Models With GraphEstimator + NSPPK

Thin notebook for fitting classical graph predictive models on persisted PCNs. It uses `NSPPK` as the graph transformer and `abstractgraph_ml.estimators.GraphEstimator` as the estimator wrapper.

## 1. Configure

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from enzyme_classification.modeling.datasets import load_graph_dataset_from_status
from enzyme_classification.representations import SimplexPairVectorizer

try:
    from abstractgraph_ml.estimators import GraphEstimator
except ImportError:
    ABSTRACTGRAPH_ML_SRC = PROJECT_ROOT.parent / "abstractgraph-ecosystem/repos/abstractgraph-ml/src"
    sys.path.insert(0, str(ABSTRACTGRAPH_ML_SRC))
    from abstractgraph_ml.estimators import GraphEstimator

try:
    from nsppk import NSPPK
except ImportError:
    NSPPK = None

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

TASK = "A"  # "A" for enzyme vs non-enzyme, "B" for first-level EC class
RANDOM_STATE = 42
TEST_SIZE = 0.20
MIN_SAMPLES_PER_CLASS_FOR_EVALUATION = 5
FILTER_UNDER_SUPPORTED_CLASSES = False
SPLIT_GROUP_COLUMN = "protein_name"  # Use "protein_name" to keep repeated/related named proteins in one split.
SPLIT_GROUP_FALLBACK_COLUMNS = ["uniprot_accessions", "entity_id", "pdb_id"]
TRANSFORMER_KIND = "simplex_pair"  # "simplex_pair" or "nsppk"


TASK_A_STATUS = PROJECT_ROOT / "data/processed/task_a/graph_build_status.tsv"
TASK_B_STATUS = PROJECT_ROOT / "data/processed/task_b/graph_build_status.tsv"

STATUS_PATH = TASK_A_STATUS if TASK.upper() == "A" else TASK_B_STATUS
RESULTS_ROOT = PROJECT_ROOT / "results" / f"task_{TASK.lower()}"
STATUS_PATH

PosixPath('/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/task_a/graph_build_status.tsv')

## 2. Load Persisted Graph Dataset

In [2]:
dataset = load_graph_dataset_from_status(STATUS_PATH, task=TASK)
graphs = dataset.graphs
y = dataset.labels
metadata = dataset.metadata

print(f"task={TASK}")
print(f"status_path={STATUS_PATH}")
print(f"graphs={len(graphs)} labels={len(y)}")
metadata[["sample_id", "pdb_id", "protein_name", "first_level_ec_numbers", "target", "graph_path"]].head()

task=A
status_path=/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/task_a/graph_build_status.tsv
graphs=520 labels=520


,sample_id,pdb_id,protein_name,first_level_ec_numbers,target,graph_path
0,10dc.A,10dc,GTPase HRas,3,enzyme,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
1,10dj.A,10dj,Tyrosine-protein kinase Fyn,2,enzyme,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
2,10dj.B,10dj,Tyrosine-protein kinase Fyn,2,enzyme,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
3,10en.A,10en,"Isoform 4 of Glutamate receptor ionotropic, NM...",,non_enzyme,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
4,10en.C,10en,"Isoform 4 of Glutamate receptor ionotropic, NM...",,non_enzyme,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...


In [3]:
pd.Series(y, name="target").value_counts().sort_index()

target
enzyme        218
non_enzyme    302
Name: count, dtype: int64

## 3. Train/Test Split

In [4]:
if len(graphs) < 2:
    raise ValueError("Need at least two completed graphs before training.")

all_labels = sorted(pd.Series(y).unique())
class_counts = pd.Series(y).value_counts().sort_index()
under_supported_classes = class_counts[class_counts < MIN_SAMPLES_PER_CLASS_FOR_EVALUATION]
print("class support")
print(class_counts)

if not under_supported_classes.empty:
    print(
        "\nSmoke-test warning: these classes have fewer than "
        f"{MIN_SAMPLES_PER_CLASS_FOR_EVALUATION} samples and cannot support a reliable held-out evaluation."
    )
    print(under_supported_classes)

if FILTER_UNDER_SUPPORTED_CLASSES and not under_supported_classes.empty:
    keep_labels = set(class_counts[class_counts >= MIN_SAMPLES_PER_CLASS_FOR_EVALUATION].index)
    keep_idx = [i for i, label in enumerate(y) if label in keep_labels]
    graphs_for_split = [graphs[i] for i in keep_idx]
    y_for_split = [y[i] for i in keep_idx]
    metadata_for_split = metadata.iloc[keep_idx].reset_index(drop=True)
    all_labels = sorted(keep_labels)
    print(f"\nFiltered to classes with >= {MIN_SAMPLES_PER_CLASS_FOR_EVALUATION} samples: {all_labels}")
else:
    graphs_for_split = graphs
    y_for_split = y
    metadata_for_split = metadata.reset_index(drop=True)

def make_split_group(row):
    columns = [SPLIT_GROUP_COLUMN, *SPLIT_GROUP_FALLBACK_COLUMNS]
    for column in columns:
        value = str(row.get(column, "")).strip()
        if value:
            normalized = ";".join(sorted(part.strip().lower() for part in value.split(";") if part.strip()))
            return f"{column}:{normalized}"
    return f"sample_id:{row.get('sample_id', row.name)}"

metadata_for_split = metadata_for_split.copy()
metadata_for_split["split_group"] = metadata_for_split.apply(make_split_group, axis=1)
groups_for_split = metadata_for_split["split_group"].tolist()

split_counts = pd.Series(y_for_split).value_counts().sort_index()
active_under_supported_classes = split_counts[split_counts < MIN_SAMPLES_PER_CLASS_FOR_EVALUATION]
group_counts = pd.Series(groups_for_split).value_counts()

if len(graphs_for_split) < 2 or len(split_counts) < 2:
    raise ValueError("Need at least two classes and two completed graphs after filtering before training.")
if group_counts.size < 2:
    raise ValueError("Need at least two leakage groups for a grouped train/test split.")

split_strategy = "group_shuffle"
stratified = False
indices = np.arange(len(graphs_for_split))
y_array = np.asarray(y_for_split)
groups_array = np.asarray(groups_for_split)

candidate_splits = []
n_splits = min(max(2, round(1 / TEST_SIZE)), int(group_counts.size), int(split_counts.min()))
if n_splits >= 2:
    try:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
        candidate_splits = list(splitter.split(indices, y_array, groups_array))
        split_strategy = f"stratified_group_kfold_{n_splits}"
        stratified = True
    except ValueError as error:
        print(f"\nSplit warning: StratifiedGroupKFold failed ({error}); falling back to GroupShuffleSplit.")

if not candidate_splits:
    splitter = GroupShuffleSplit(n_splits=50, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    candidate_splits = list(splitter.split(indices, y_array, groups_array))
    split_strategy = "group_shuffle"
    stratified = False

def split_score(split):
    train_candidate, test_candidate = split
    test_labels = set(y_array[test_candidate])
    missing_count = len(set(all_labels) - test_labels)
    test_fraction_error = abs((len(test_candidate) / len(indices)) - TEST_SIZE)
    return missing_count, test_fraction_error, len(test_candidate)

train_idx_array, test_idx_array = min(candidate_splits, key=split_score)
train_idx = train_idx_array.tolist()
test_idx = test_idx_array.tolist()

train_groups = set(groups_array[train_idx])
test_groups = set(groups_array[test_idx])
group_overlap = sorted(train_groups.intersection(test_groups))
if group_overlap:
    raise RuntimeError(f"Grouped split leaked {len(group_overlap)} groups across train/test.")

X_train = [graphs_for_split[i] for i in train_idx]
X_test = [graphs_for_split[i] for i in test_idx]
y_train = [y_for_split[i] for i in train_idx]
y_test = [y_for_split[i] for i in test_idx]

train_labels = set(y_train)
test_labels = set(y_test)
missing_from_train = sorted(set(all_labels) - train_labels)
missing_from_test = sorted(set(all_labels) - test_labels)
VALID_HELD_OUT_EVALUATION = (
    not missing_from_train
    and not missing_from_test
    and active_under_supported_classes.empty
    and not group_overlap
)

print(
    f"\ntrain={len(X_train)} test={len(X_test)} "
    f"strategy={split_strategy} stratified={stratified} group_column={SPLIT_GROUP_COLUMN}"
)
print(f"groups: train={len(train_groups)} test={len(test_groups)} overlap={len(group_overlap)} total={group_counts.size}")
print("largest groups")
print(group_counts.head(10))
print("train labels")
print(pd.Series(y_train).value_counts().sort_index())
print("test labels")
print(pd.Series(y_test).value_counts().sort_index())

if missing_from_train or missing_from_test:
    print("\nEvaluation warning: this split is not a complete held-out multiclass evaluation.")
    if missing_from_train:
        print(f"classes missing from train: {missing_from_train}")
    if missing_from_test:
        print(f"classes missing from test: {missing_from_test}")

if not VALID_HELD_OUT_EVALUATION:
    print("Treat the metrics below as smoke-test diagnostics, not final model performance.")


class support
enzyme        218
non_enzyme    302
Name: count, dtype: int64

train=434 test=86 strategy=stratified_group_kfold_5 stratified=True group_column=protein_name
groups: train=68 test=21 overlap=0 total=89
largest groups
protein_name:e3 ubiquitin-protein ligase mylip            116
protein_name:microtubule-associated protein tau            21
protein_name:histone h3.2                                  20
protein_name:type iiib beta-amyloid 40 filament            20
protein_name:histone h4                                    20
protein_name:transitional endoplasmic reticulum atpase     18
protein_name:glutamate receptor ionotropic, nmda 2b        18
protein_name:glutamate receptor ionotropic, nmda 1         16
protein_name:glutathione s-transferase                     16
protein_name:histone h2a type 1                            14
Name: count, dtype: int64
train labels
enzyme        192
non_enzyme    242
Name: count, dtype: int64
test labels
enzyme        26
non_enzyme    60
Nam

## 4. Define GraphEstimator Transformer


In [5]:
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer


if TRANSFORMER_KIND == "nsppk":
    if NSPPK is None:
        raise ImportError("TRANSFORMER_KIND='nsppk' requires the nsppk package on PYTHONPATH.")
    transformer = NSPPK(
        radius=1,
        distance=4,
        connector=1,
        nbits=14,
        dense=False,
        parallel=True,
    )
elif TRANSFORMER_KIND == "simplex_pair":
    transformer = SimplexPairVectorizer(
        max_simplex_dimension=3,
        min_distance=0,
        max_distance=4,
        distance_mode="representative",
        distance_jitter="auto",
        pairing_mode="rooted",
        nbits=14,
        dense=False,
        include_self=True,
        cache_graphs=False,
        n_jobs=-1,
    )
else:
    raise ValueError(f"Unknown TRANSFORMER_KIND={TRANSFORMER_KIND!r}")

base_estimator = make_pipeline(
    Normalizer(norm="l2"),
    SGDClassifier(
        loss="hinge",
        alpha=1e-4,
        max_iter=5000,
        tol=1e-4,
        n_iter_no_change=20,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        average=True,
        n_jobs=-1,
    ),
)
model = GraphEstimator(
    transformer=transformer,
    estimator=base_estimator,
    manifold=None,
    n_selected_features=None,
)
model


,transformer,"SimplexPairVe...=-1, nbits=14)"
,estimator,Pipeline(step...tol=0.0001))])
,manifold,<abstractgrap...x73806b67f0e0>
,n_selected_features,None
,"norm norm: {'l1', 'l2', 'max'}, default='l2'The norm to use to normalize each non zero sample. If norm='max'is used, values will be rescaled by the maximum of the absolutevalues.",'l2'
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array or a scipy.sparseCSR matrix).",True
,"loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'hinge'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True


## 5. Fit

In [6]:
model.fit(X_train, y_train)
model

,transformer,"SimplexPairVe...=-1, nbits=14)"
,estimator,Pipeline(step...tol=0.0001))])
,manifold,<abstractgrap...x73806b67f0e0>
,n_selected_features,None
,"norm norm: {'l1', 'l2', 'max'}, default='l2'The norm to use to normalize each non zero sample. If norm='max'is used, values will be rescaled by the maximum of the absolutevalues.",'l2'
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array or a scipy.sparseCSR matrix).",True
,"loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'hinge'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True


## 6. Test Metrics

In [7]:
y_pred = model.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "valid_held_out_evaluation": VALID_HELD_OUT_EVALUATION,
}
metrics_df = pd.Series(metrics, name="value").to_frame()
metrics_df

,value
accuracy,0.965116
balanced_accuracy,0.953205
valid_held_out_evaluation,True


In [8]:
report_labels = all_labels
report = classification_report(
    y_test,
    y_pred,
    labels=report_labels,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T
report_df

,precision,recall,f1-score,support
enzyme,0.960000,0.923077,0.941176,26.000000
non_enzyme,0.967213,0.983333,0.975207,60.000000
accuracy,0.965116,0.965116,0.965116,0.965116
macro avg,0.963607,0.953205,0.958192,86.000000
weighted avg,0.965032,0.965116,0.964918,86.000000


In [9]:
cm = confusion_matrix(y_test, y_pred, labels=report_labels)
confusion_matrix_df = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in report_labels],
    columns=[f"pred_{label}" for label in report_labels],
)
confusion_matrix_df

,pred_enzyme,pred_non_enzyme
true_enzyme,24,2
true_non_enzyme,1,59


## 7. Inspect Predictions

In [10]:
predictions = metadata_for_split.iloc[test_idx][["sample_id", "pdb_id", "protein_name", "split_group", "first_level_ec_numbers", "graph_path"]].copy()
predictions["y_true"] = y_test
predictions["y_pred"] = y_pred
predictions["correct"] = predictions["y_true"] == predictions["y_pred"]
predictions.sort_values(["correct", "sample_id"]).head(30)

,sample_id,pdb_id,protein_name,split_group,first_level_ec_numbers,graph_path,y_true,y_pred,correct
199,10ft.A,10ft,Protein kinase C-binding protein NELL2,protein_name:protein kinase c-binding protein ...,,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...,non_enzyme,enzyme,False
518,1aax.A,1aax,PROTEIN TYROSINE PHOSPHATASE 1B,protein_name:protein tyrosine phosphatase 1b,3,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...,enzyme,non_enzyme,False
517,1ab2.A,1ab2,C-ABL TYROSINE KINASE SH2 DOMAIN,protein_name:c-abl tyrosine kinase sh2 domain,2,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...,enzyme,non_enzyme,False
0,10dc.A,10dc,GTPase HRas,protein_name:gtpase hras,3,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...,enzyme,enzyme,True
5,10en.B,10en,Glutamate receptor,protein_name:glutamate receptor,,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...,non_enzyme,non_enzyme,True
6,10en.D,10en,Glutamate receptor,protein_name:glutamate receptor,,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...,non_enzyme,non_enzyme,True
234,10ev.B,10ev,Glutamate receptor,protein_name:glutamate receptor,,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...,non_enzyme,non_enzyme,True
235,10ev.D,10ev,Glutamate receptor,protein_name:glutamate receptor,,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...,non_enzyme,non_enzyme,True
197,10gs.A,10gs,GLUTATHIONE S-TRANSFERASE P1-1,protein_name:glutathione s-transferase p1-1,2,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...,enzyme,enzyme,True
198,10gs.B,10gs,GLUTATHIONE S-TRANSFERASE P1-1,protein_name:glutathione s-transferase p1-1,2,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...,enzyme,enzyme,True


## 8. Persist Results

In [11]:
run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_name = f"{run_timestamp}_graph_estimator_{TRANSFORMER_KIND}"
run_dir = RESULTS_ROOT / run_name
run_dir.mkdir(parents=True, exist_ok=False)

predictions_path = run_dir / "predictions.tsv"
metrics_path = run_dir / "metrics.tsv"
report_path = run_dir / "classification_report.tsv"
confusion_matrix_path = run_dir / "confusion_matrix.tsv"
metadata_path = run_dir / "run_metadata.json"

predictions.to_csv(predictions_path, sep="	", index=False)
metrics_df.to_csv(metrics_path, sep="	")
report_df.to_csv(report_path, sep="	")
confusion_matrix_df.to_csv(confusion_matrix_path, sep="	")

run_metadata = {
    "run_name": run_name,
    "created_at_utc": run_timestamp,
    "task": TASK,
    "status_path": str(STATUS_PATH),
    "results_dir": str(run_dir),
    "n_graphs_loaded": len(graphs),
    "n_graphs_used": len(graphs_for_split),
    "n_train": len(X_train),
    "n_test": len(X_test),
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "split_strategy": split_strategy,
    "split_group_column": SPLIT_GROUP_COLUMN,
    "split_group_fallback_columns": SPLIT_GROUP_FALLBACK_COLUMNS,
    "n_split_groups": int(group_counts.size),
    "n_train_groups": len(train_groups),
    "n_test_groups": len(test_groups),
    "group_overlap_count": len(group_overlap),
    "stratified": stratified,
    "all_labels": [str(label) for label in all_labels],
    "class_counts": {str(k): int(v) for k, v in class_counts.items()},
    "active_class_counts": {str(k): int(v) for k, v in split_counts.items()},
    "train_class_counts": {str(k): int(v) for k, v in pd.Series(y_train).value_counts().sort_index().items()},
    "test_class_counts": {str(k): int(v) for k, v in pd.Series(y_test).value_counts().sort_index().items()},
    "missing_from_train": [str(label) for label in missing_from_train],
    "missing_from_test": [str(label) for label in missing_from_test],
    "valid_held_out_evaluation": bool(VALID_HELD_OUT_EVALUATION),
    "min_samples_per_class_for_evaluation": MIN_SAMPLES_PER_CLASS_FOR_EVALUATION,
    "filter_under_supported_classes": FILTER_UNDER_SUPPORTED_CLASSES,
    "model": "GraphEstimator",
    "transformer": TRANSFORMER_KIND,
    "transformer_class": type(transformer).__name__,
    "transformer_params": transformer.get_params() if hasattr(transformer, "get_params") else {},
    "estimator": type(model.estimator).__name__ if hasattr(model, "estimator") else "unknown",
    "estimator_repr": repr(model.estimator) if hasattr(model, "estimator") else "unknown",
    "python_version": platform.python_version(),
}
with metadata_path.open("w", encoding="utf-8") as handle:
    json.dump(run_metadata, handle, indent=2, sort_keys=True)

saved_files = pd.DataFrame(
    {
        "artifact": ["predictions", "metrics", "classification_report", "confusion_matrix", "run_metadata"],
        "path": [predictions_path, metrics_path, report_path, confusion_matrix_path, metadata_path],
    }
)
saved_files

,artifact,path
0,predictions,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
1,metrics,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
2,classification_report,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
3,confusion_matrix,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
4,run_metadata,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
